In [1]:
import torch
import sys

sys.path.append('../model')

In [2]:
from token_classifier_base import TokenClassifier, TokenClassifierConfig
from modules import SinPositionalEncoding

In [3]:
class RecycleModule(torch.nn.Module):
    def __init__(self, classifier_body  : torch.nn.Module, n_steps, input_dim):
        self.body = classifier_body
        self.n_steps = n_steps

        self.input_dim = input_dim
        self.register_buffer("initial_state", torch.zeros(1, 1, self.input_dim))

    def iteration(self, inputs, attention_mask=None):
        B, L, D = inputs.shape
        
        # Initialize prev_f with zeros
        prev_f = self.initial_state.expand(B, L, -1)
        
        for i in range(self.num_recycles + 1):
            # AF2 Style: Detach gradients for all passes except the final one
            # This allows for deep 'pseudo-depth' without massive memory costs
            if self.training and i < self.num_recycles:
                prev_f = prev_f.detach()
            
            # Combine embeddings with previous refinement (Addition or Concat)
            current_input = inputs + prev_f
            
            # Refine features
            # mask ensures we ignore padding tokens in the Transformer
            prev_f = self.body(current_input, src_key_padding_mask=attention_mask)
        
        # Return refined state
        return prev_f

In [31]:
class ResidualTransformerLayer(torch.nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.self_attn = torch.nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        
        # Implementation of Feedforward block
        self.linear1 = torch.nn.Linear(d_model, dim_feedforward)
        self.dropout = torch.nn.Dropout(dropout)
        self.linear2 = torch.nn.Linear(dim_feedforward, d_model)

        # Normalization layers
        self.norm1 = torch.nn.LayerNorm(d_model)
        self.norm2 = torch.nn.LayerNorm(d_model)
        self.dropout1 = torch.nn.Dropout(dropout)
        self.dropout2 = torch.nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # 1. Multi-head Attention + Residual (Pre-LN style)
        # We normalize BEFORE the sub-layer and add the result back to the original input
        residual = x
        x = self.norm1(x)

        # Some vals will become none otherwise, taken from ESM-2 repo
        if mask is not None:
            mask = mask.to(torch.float32)
            mask = mask.masked_fill(
                mask.to(torch.bool), -1e8 if x.dtype == torch.float32 else -1e4
            )
        attn_output, _ = self.self_attn(x, x, x, key_padding_mask=mask)
        x = residual + self.dropout1(attn_output)

        # 2. Feedforward + Residual (Pre-LN style)
        residual = x
        x = self.norm2(x)
        ff_output = self.linear2(self.dropout(torch.relu(self.linear1(x))))
        x = residual + self.dropout2(ff_output)
        
        return x
    
class RecyclingEncoder(torch.nn.Module):
    def __init__(self, d_model, nhead, num_layers, num_recycles, dropout, d_feedforward):
        super().__init__()
        self.num_recycles = num_recycles
        self.layers = torch.nn.ModuleList([
            ResidualTransformerLayer(d_model, nhead, dropout=dropout, dim_feedforward=d_feedforward) for _ in range(num_layers)
        ])
        self.norm_final = torch.nn.LayerNorm(d_model)
        self.initial_state = torch.nn.Parameter(torch.zeros(d_model))

    def forward(self, inputs, mask=None):
        # Initial 'prev_f' state (could be zeros or a copy of x)
        prev_f = self.initial_state
        for r in range(self.num_recycles + 1):
            # AF2 Gradient Detachment: Only the last cycle contributes to training
            if self.training and r < self.num_recycles:
                prev_f = prev_f.detach()

            # The current state is the sum of base ESM embeddings and previous refinements
            out = inputs + prev_f 
            
            for layer in self.layers:
                out = layer(out, mask=mask)
            
            prev_f = out # Update state for next recycle
            
        return self.norm_final(prev_f)

In [21]:
from dataclasses import dataclass

@dataclass
class RecyclingClassifierConfig(TokenClassifierConfig):
    n_recycle_steps : int = 3
    dim_model : int = 256
    dim_ffw : int = 2048
    n_heads : int = 8
    n_enc_layers : int = 3
    

In [22]:
class RecyclingClassifier(TokenClassifier):
    def __init__(self, base_model, config):
        super().__init__(config, base_model)
        
        # Pos embeds from ESM-2 already in the embeddings
        # self.pos_embed = SinPositionalEncoding(config.dim_model, 1024) # ESM has max 1024 tokens, incl. [cls]...[eos]
        model_dim = base_model.config.hidden_size
        self.encoder = RecyclingEncoder(model_dim, config.n_heads, config.n_enc_layers, config.n_recycle_steps,
                                        dropout=config.dropout_rate, d_feedforward=config.dim_ffw)
        self.output = torch.nn.Linear(model_dim, config.n_labels)

    def forward(self, input_ids, attention_mask, **kwargs):
        base_out = self.base(input_ids=input_ids, attention_mask=attention_mask)
        x = base_out[0]
        # x = x + self.pos_embed(x)
        if 'no_flash_attn' in kwargs and kwargs['no_flash_attn']:
            # Transform the inputs to sequence-first. Expecting batch size of 1
            x = x.moveaxis(0, 1).squeeze()
            x = self.encoder(x)
            x = x.unsqueeze(0)
        else:
            x = self.encoder(x, mask=attention_mask)

        return self.output(x), base_out

In [23]:
from esm_train import get_esm
from training import create_loss, run_training, parser
from argparse import Namespace

def create_model(args):
    esm, tokenizer = get_esm(args.type)

    config = RecyclingClassifierConfig(n_labels=1,loss=create_loss(args), base_type=args.type, n_recycle_steps=args.n_recycle_steps,
                                       n_heads = args.n_heads, n_enc_layers = args.n_enc_layers, dropout_rate=args.dropout,
                                       dim_ffw=args.dim_ffw)
    model = RecyclingClassifier(base_model=esm, config=config)
    model.set_base_requires_grad(False)
    return model, tokenizer

def add_arguments(parser):
    parser.add_argument('--n_heads', type=int, default=8)
    parser.add_argument('--n_recycle_steps', type=int, default=3)
    parser.add_argument('--n_enc_layers', type=int, default=3)
    parser.add_argument('--dim_ffw', type=int, default=512)


In [ ]:
add_arguments(parser)

In [25]:
args = parser.parse_args([])

In [26]:
args.type = '35M'
args.epochs=10
args.dataset_path = '../data/dbptm/splits_Y.json'
args.prot_info_path = '../data/dbptm/dbptm_info.json'
args.dropout = 0.1

In [27]:
args

Namespace(seed=42, batch_size=4, epochs=10, prot_info_path='../data/dbptm/dbptm_info.json', dataset_path='../data/dbptm/splits_Y.json', weight_decay=0.0001, accum=3, hidden_size=128, lr=0.0003, o=None, n='esm', compile=False, lora=False, dropout=0.1, type='35M', pos_weight=3, num_workers=0, n_layers=1, checkpoint_path=None, model_path=None, focal=False, residues="['S', 'T', 'Y']", ignore_label=-1, patience=20, debug=False, step_lr=False, release=False, modify_prob=0, mask_prob=0.7, rand_prob=0.15, sub_prob=0.15, n_heads=8, n_recycle_steps=3, n_enc_layers=3, dim_ffw=512)

In [32]:
run_training(args, create_model)

Seed set to 42


Current fold: 0
Dev created from the train partition
Train size: 2052
Dev size: 513
Test size: 642


Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t12_35M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
c:\Users\Samo\repos\phossil\venv\Lib\site-packages\lightning\pytorch\callbacks\model_checkpoint.py:881: Checkpoint directory C:\Users\Samo\repos\phossil\notebooks\new_logs\training.py_2026_04_17_162632\fold_0 exists and is not empty.

  | Name               | Type                       | Params | Mode  | FLOPs
----------------------------------------------------------------------------------
0 | classifier         | RecyclingClassifier        | 37.8 M | train | 0    
1 | step_metrics       | MetricCollection           | 0      | train | 0    
2 | test_step_metrics  | MetricCollection           | 0      | train | 0    
3 | val_step_

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\Samo\repos\phossil\venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
c:\Users\Samo\repos\phossil\venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
c:\Users\Samo\repos\phossil\venv\Lib\site-packages\lightning\pytorch\loops\fit_loop.py:534: Found 217 module(s) in eval mode at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can ignore this warning.


Training: |          | 0/? [00:00<?, ?it/s]

tensor(1.0437, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(1.1248, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(1.1061, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(1.4374, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(4.0248, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(2.3701, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(2.0408, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(3.7304, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(1.7518, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(1.3044, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.7513, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(1.6764, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(1.1305, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(1.1034, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(1.1047, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(1.4038, grad_fn=<BinaryCrossEntro


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

c:\Users\Samo\repos\phossil\venv\Lib\site-packages\IPython\core\interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
